In [0]:
from pyspark.sql import Row

# # BU_Group="COMMERCIALS"
# bu_filter = ['COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL','COMNA-NORTH AMERICA','COMNN-NETHERLANDS','COMON-OCEANIA','COMSE-BELGIUM, LUXEMBOURG','COMSPB-SPAIN, PORTUGAL, BRAZIL','COMUI-UK & IRELAND','FRANC-FRANCE','ITALY-ITALY','CREDIT INSURANCE' ]

# In Progress BU
bu_filter = ['COMNA-North America']

# # BU	cnt
# # FRANC-France	60
# # COMSPB-Spain, Portugal, Brazil	84
# # COMON-Oceania	216
# # COMNA-North America	351
# # COMAS-Asia	370
# # GLOBA-Global	407
# # ITALY-Italy	447
# # COMNN-Netherlands	629
# # COMUI-UK & Ireland	919
# # COMSE-Belgium, Luxembourg	1817
# # COMCE-Germany,Austria, Switzerl,CEE	1964

# # Controller BU scanned list
# bu_filter = ['FRANC-France','COMSPB-Spain, Portugal, Brazil','COMON-Oceania']


# # Completed Extraction BUs
# bu_filter = ['RISK6-RS6-CEE and ITA','GMC-GROUP MARKETING & COMMUNICATION','GRC&R-GROUP CLAIMS & RECOVERIES','FINPM-FINANCE PROGRAM MANAGEMENT','GFO-GROUP FINANCE OPERATIONS','GFC-GROUP FINANCE AND CONTROL','COFIN-CORPORATE FINANCE & TAX','Group Finance','Finance','Finance & Control','RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE','RISK2-RS2-BELGIUM, LUX, FRANCE & ITA','RISK3-RS3-NLD, APAC & AIM','RISK4-RS4-NORTH EUROPE, CIS & ACS','RISK4-RS4-NORTH EUROPE, CIS & SP','RISK5-RS5-NAFTA','RISK1-RS1-DEU, AUT and CHE','RISK1-RS1-Europe, Russia & CIS','RISK2-RS2-NLD, Belux, FRA, Africa & ISR','RISK3-RS3-Asia and Oceania','RISK7-RS7-Spain, Portugal, Andorra']

bu_df = spark.createDataFrame([Row(bu=b) for b in bu_filter])
bu_df.createOrReplaceTempView("vw_bu_filter")

print(bu_filter)

In [0]:
%sql
with complete_list as (
  -- Owner Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(
    case 
      when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
      else cms1.created
    end
  ))
  where 
  uaa1.WEBI_Doc_ID is not null
  and
  upper(trim(ud1.user_ID)) in (select distinct upper(trim(user_ID)) from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team )
  and
  upper(trim(ud1.BU)) in (select upper(trim(bu)) from vw_bu_filter )


  Union all
  -- Viewing Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
  where uaa1.WEBI_Doc_ID is not null
  and
  upper(trim(ud1.user_ID)) in (select distinct upper(trim(user_ID)) from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team )
  and 
  upper(trim(ud1.BU)) in (select upper(trim(bu)) from vw_bu_filter )
)
select distinct Document_Id from complete_list
where 
Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
and 
Document_Id not in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary)

limit 250
-- group by BU
-- order by cnt asc

In [0]:
%sql
  select distinct cms1.Document_Id, ud1.BU, ud1.full_name from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(
    case 
      when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
      else cms1.created
    end
  ))
  where 
  uaa1.WEBI_Doc_ID is not null
  and
  upper(trim(ud1.user_ID)) in (select distinct upper(trim(user_ID)) from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team )
  and cms1.Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)

select count(distinct Document_Id ) from _sqldf

  

In [0]:
%sql
select distinct ad1.BU from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ad1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as ua1
on upper(trim(ua1.UserName))=upper(trim(ad1.user_ID))
where ua1.UserName is not null
and upper(trim(ad1.BU)) not in (select upper(trim(bu)) from vw_bu_filter)
order by ad1.BU asc